In [1]:
# PFE Renault Tanger — Système d'alertes
## Notebook 05 : Alertes email automatiques + SHAP
### Objectif : envoyer un rapport quotidien intelligent par email

In [2]:
import pandas as pd
import numpy as np
import smtplib
import joblib
import shap
import matplotlib.pyplot as plt
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.image import MIMEImage
from datetime import datetime, date
import warnings
warnings.filterwarnings('ignore')

print("Librairies chargées ✓")

Librairies chargées ✓


In [3]:
import requests
import os
from datetime import datetime

DRIVE_FILE_ID = "1wR7e7dtbgBmnu5xDTB0kHpo9IpNsrUbp"  
FICHIER_LOCAL = "../data/Synthèse_Eaux_2026_VF_(2).xlsm"

def telecharger_depuis_drive():
    """
    Télécharge un Google Sheets en format Excel (.xlsx)
    """
    # URL d'export Google Sheets → Excel
    url = f"https://docs.google.com/spreadsheets/d/{DRIVE_FILE_ID}/export?format=xlsx"
    
    print(f"[Drive] Téléchargement en cours...")
    print(f"[Drive] Heure : {datetime.now().strftime('%H:%M:%S')}")
    
    response = requests.get(url)
    
    if response.status_code == 200:
        with open(FICHIER_LOCAL, 'wb') as f:
            f.write(response.content)
        taille = os.path.getsize(FICHIER_LOCAL) / 1024
        print(f"[Drive] Fichier sauvegardé ✓")
        print(f"[Drive] Taille : {taille:.1f} Ko")
    else:
        print(f"[Drive] Erreur {response.status_code}")
        print("[Drive] Vérifie que le fichier est partagé publiquement")

telecharger_depuis_drive()

[Drive] Téléchargement en cours...
[Drive] Heure : 22:39:46
[Drive] Fichier sauvegardé ✓
[Drive] Taille : 579.7 Ko


In [4]:
import os
print("Dossier actuel :", os.getcwd())
print("Contenu de ../scripts :", os.listdir("../scripts"))

Dossier actuel : /Users/akrambelhaj/Desktop/PFE_Renault/notebooks
Contenu de ../scripts : ['.DS_Store', '__pycache__', 'pipeline_etl_eau_renault_1.py', '.ipynb_checkpoints']


In [5]:
import subprocess
import os  
import sys
print("[ETL] Mise à jour du dataset...")

# Supprimer l'ancien CSV pour forcer rechargement complet
if os.path.exists("../outputs/dataset_eau_propre.csv"):
    os.remove("../outputs/dataset_eau_propre.csv")
    print("[ETL] Ancien dataset supprimé ✓")

# Relancer le pipeline ETL
import sys
sys.path.append("../scripts")
import pipeline_etl_eau_renault_1 as etl

etl.FICHIER_EXCEL  = "../data/Synthèse_Eaux_2026_VF_(2).xlsm"
etl.FICHIER_SORTIE = "../outputs/dataset_eau_propre.csv"
df_nouveau = etl.run_pipeline()

print(f"[ETL] Dataset mis à jour jusqu'au {df_nouveau['Date'].max().date()} ✓")

[ETL] Mise à jour du dataset...
[ETL] Ancien dataset supprimé ✓
 Pipeline ETL Eau - Renault Tanger
 Exécution : 2026-05-21 22:39:51
[ETL] Lecture du fichier : ../data/Synthèse_Eaux_2026_VF_(2).xlsm
[ETL] 140 lignes trouvées (de 2026-01-01 à 2026-05-20)
[ETL] Nettoyage des données...
  → ED_Total : 1 zéros remplacés par NaN
  → EOR_Total : 1 zéros remplacés par NaN
  → EP_Looker : 2 zéros remplacés par NaN
[ETL] Création des features...
  → 65 colonnes dans le dataset final
  → Jours normaux (TCM≥100) : 109
  → Jours arrêt                        : 31
  → Anomalies KPI détectées            : 18
  → Anomalies recyclage détectées      : 16
[ETL] Chargement du dataset...
[ETL] Fichier créé : ../outputs/dataset_eau_propre.csv (140 lignes)
 Pipeline terminé avec succès ✓
[ETL] Dataset mis à jour jusqu'au 2026-05-20 ✓


In [6]:
!pip install gspread google-auth pandas

In [7]:
#Exporter le dataset vers Google Sheets

import gspread
from google.oauth2.service_account import Credentials
import pandas as pd


SCOPES = [
    'https://www.googleapis.com/auth/spreadsheets',
    'https://www.googleapis.com/auth/drive'
]

# Authentification avec le fichier credentials.json
creds   = Credentials.from_service_account_file(
    '../config/credentials.json',
    scopes=SCOPES
)
client = gspread.authorize(creds)

# Ouvrir  le Google Sheets
SHEETS_ID = "1an9bSKsZl7AqPTQiLyvhbO50X6QhPsZDhwKU7Nj2hkE"
sheet     = client.open_by_key(SHEETS_ID)
print(f"Google Sheets ouvert ✓")

# Sélectionner la feuille principale
worksheet = sheet.get_worksheet(0)
worksheet.clear()

# Charger le dataset propre
df = pd.read_csv("../outputs/dataset_eau_propre.csv")

# Convertir et envoyer vers Google Sheets
df['Date'] = df['Date'].astype(str)
df = df.fillna(0)

# Envoyer les headers + données
data = [df.columns.tolist()] + df.values.tolist()
worksheet.update(data)

print(f"✅ {len(df)} lignes envoyées vers Google Sheets")
print(f"   Sheets : {sheet.title}")

# Récupérer l'URL du Sheets
sheet_url = f"https://docs.google.com/spreadsheets/d/{sheet.id}"
print(f"   URL : {sheet_url}")

Google Sheets ouvert ✓
✅ 140 lignes envoyées vers Google Sheets
   Sheets : PFE_Renault_Dashboard_Eau
   URL : https://docs.google.com/spreadsheets/d/1an9bSKsZl7AqPTQiLyvhbO50X6QhPsZDhwKU7Nj2hkE


In [8]:
import gspread
import pandas as pd
import numpy as np
import time
from google.oauth2.service_account import Credentials
 
# Connexion (réutilise la connexion déjà établie en cellule 7)
# Si tu relances uniquement cette cellule, décommente les lignes suivantes :
# SCOPES = ["https://www.googleapis.com/auth/spreadsheets",
#           "https://www.googleapis.com/auth/drive"]
# creds  = Credentials.from_service_account_file("../config/credentials.json", scopes=SCOPES)
# client = gspread.authorize(creds)
# sheet  = client.open_by_key("1an9bSKsZl7AqPTQiLyvhbO50X6QhPsZDhwKU7Nj2hkE")
 
print("[Sheets] Export Feuille 1 en cours...")
 
worksheet_data = sheet.get_worksheet(0)
worksheet_data.clear()
 
# Chargement et nettoyage des données
df_export = pd.read_csv("../outputs/dataset_eau_propre.csv")
df_export['Date'] = df_export['Date'].astype(str)
df_export = df_export.replace([np.inf, -np.inf], 0).fillna(0).round(3)
 
# Préparation des données pour l'envoi
header       = df_export.columns.tolist()
values       = df_export.values.tolist()
data_to_send = [header] + values
 
# Envoi par blocs de 1000 lignes avec retry automatique
rows_per_chunk = 1000
 
for i in range(0, len(data_to_send), rows_per_chunk):
    chunk     = data_to_send[i : i + rows_per_chunk]
    start_row = i + 1
    max_retries = 3
 
    for attempt in range(max_retries):
        try:
            worksheet_data.update(chunk, range_name=f"A{start_row}")
            print(f"  Bloc {i // rows_per_chunk + 1} envoyé avec succès...")
            break
        except Exception as e:
            print(f"  [!] Coupure réseau sur le bloc {i // rows_per_chunk + 1}. "
                  f"Tentative {attempt + 1}/{max_retries}...")
            time.sleep(5)
            if attempt == max_retries - 1:
                print("  [ERREUR FATALE] Impossible d'envoyer ce bloc après 3 tentatives.")
                raise e
 
    time.sleep(2)
 
print(f"[Sheets] Mise à jour terminée ✓ ({len(df_export)} lignes exportées)")

[Sheets] Export Feuille 1 en cours...
  Bloc 1 envoyé avec succès...
[Sheets] Mise à jour terminée ✓ (140 lignes exportées)


In [9]:
# Ajoute cette cellule après la création du Sheets
sheet.share(
    'akrambelhaj55l@gmail.com',
    perm_type='user',
    role='writer'
)
print("Sheets partagé avec ton compte ✓")

Sheets partagé avec ton compte ✓


In [10]:

CONFIG_EMAIL = {
    'expediteur'     : 'akrambelhaj55@gmail.com',
    'mot_de_passe'   : 'ttro oydp uwtd vquo',   
    'destinataires'  : [
        'nisrineelmoubariki5@gmail.com',
        'akrambelhaj55@gmail.com'
    ],
    'smtp_serveur'   : 'smtp.gmail.com',
    'smtp_port'      : 587
}

SEUIL_OBJECTIF = 1.25   # m³/véhicule
SEUIL_ALERTE   = 1.25 * 1.15   # +15% = anomalie

print("Configuration chargée ✓")


Configuration chargée ✓


In [11]:
df = pd.read_csv("../outputs/dataset_eau_propre.csv", parse_dates=['Date'])
df_prod = df[(df['TCM'] >= 100) & (df['is_weekend'] == 0)].copy().reset_index(drop=True)

model_xgb = joblib.load("../models/model_xgboost.pkl")

# ── CORRECTION Prophet : réentraîner avec données filtrées ──
from prophet import Prophet

df_prophet = df_prod[['Date','KPI_m3_veh','TCM']].copy()
df_prophet.columns = ['ds','y','TCM']
df_prophet = df_prophet.dropna(subset=['y'])

# Filtrer les outliers extrêmes (KPI > 3 = jours aberrants janvier)
df_prophet = df_prophet[df_prophet['y'] < 3.0].copy()

# Réentraîner Prophet sur données propres
model_prophet = Prophet(
    yearly_seasonality=False,   # pas assez de données pour annuelle
    weekly_seasonality=True,
    daily_seasonality=False,
    seasonality_mode='additive', # additif = pas de valeurs négatives
    changepoint_prior_scale=0.01 # très stable, peu de flexibilité
)
model_prophet.add_regressor('TCM')
model_prophet.fit(df_prophet)

# Recalculer SHAP
FEATURES = [
    'TCM', 'ED_Total', 'EOR_Total', 'EI_Facturation', 'EP_Facturation',
    'jour_semaine', 'mois', 'trimestre', 'is_lundi',
    'ratio_EOR_ED', 'taux_EP', 'taux_EI'
]
df_xgb  = df_prod[FEATURES + ['KPI_m3_veh', 'Date', 'E_Appro Facturation']].dropna().copy()
X       = df_xgb[FEATURES]

explainer   = shap.TreeExplainer(model_xgb)
shap_values = explainer.shap_values(X)

# Sauvegarder le nouveau Prophet
joblib.dump(model_prophet, "../models/model_prophet.pkl")

print("Données et modèles chargés ✓")
print(f"Données Prophet filtrées : {len(df_prophet)} jours (KPI < 3.0)")
print(f"Dernier jour disponible  : {df_xgb['Date'].max().date()}")

Importing plotly failed. Interactive plots will not work.
22:40:01 - cmdstanpy - INFO - Chain [1] start processing
22:40:01 - cmdstanpy - INFO - Chain [1] done processing


Données et modèles chargés ✓
Données Prophet filtrées : 85 jours (KPI < 3.0)
Dernier jour disponible  : 2026-05-20


In [12]:
import pandas as pd
import numpy as np
from datetime import date

def safe_float(val):
    try:
        f = float(val)
        return f if not (np.isnan(f) or np.isinf(f)) else np.nan
    except:
        return np.nan

def calculer_kpis_dashboard(df_prod, model_prophet):
    today      = pd.Timestamp(date.today())
    debut_ann  = pd.Timestamp(f"{today.year}-01-01")
    debut_mois = pd.Timestamp(f"{today.year}-{today.month:02d}-01")

    df_kpi = df_prod[df_prod['TCM'] >= 100].copy()
    df_kpi = df_kpi.sort_values('Date').reset_index(drop=True)

    fichier_excel = "../data/Synthèse_Eaux_2026_VF_(2).xlsm"

    # ── KPI YTD + MTD depuis Synthèse_mois ──────────────────
    df_mois_raw = pd.read_excel(
        fichier_excel,
        sheet_name='Synthèse_mois',
        header=None
    )

    # KPI YTD = E_Appro Total Général / TCM Total Général
    ligne_total = df_mois_raw.iloc[16]
    tcm_total   = safe_float(ligne_total[1])
    appro_total = safe_float(ligne_total[2])

    if tcm_total and appro_total and tcm_total > 0:
        kpi_ytd = round(appro_total / tcm_total, 3)
        print(f"KPI YTD : {appro_total:.0f} / {tcm_total:.0f} = {kpi_ytd:.3f} ✅")
    else:
        print("⚠️  Total général invalide → calcul Python")
        df_ytd_sec = df_kpi[df_kpi['Date'] >= debut_ann]
        kpi_ytd = round(float(df_ytd_sec['KPI_m3_veh'].mean()), 3)

    # KPI MTD = E_Appro mois actuel / TCM mois actuel
    mois_actuel = today.month
    idx_mois    = 3 + mois_actuel
    ligne_mois  = df_mois_raw.iloc[idx_mois]
    tcm_mois    = safe_float(ligne_mois[1])
    appro_mois  = safe_float(ligne_mois[2])
    nom_mois    = str(ligne_mois[0])

    if tcm_mois and appro_mois and tcm_mois > 0:
        kpi_mtd = round(appro_mois / tcm_mois, 3)
        print(f"KPI MTD : {appro_mois:.0f} / {tcm_mois:.0f} = {kpi_mtd:.3f} ({nom_mois}) ✅")
    else:
        print(f"⚠️  Mois {nom_mois} invalide → calcul Python")
        df_mtd_sec = df_kpi[df_kpi['Date'] >= debut_mois]
        kpi_mtd = round(float(df_mtd_sec['KPI_m3_veh'].mean()), 3) \
                  if not df_mtd_sec.empty else np.nan

    # ── CUMUL YTD ────────────────────────────────────────────
    cumul_ytd = df_kpi[
        df_kpi['Date'] >= debut_ann
    ]['E_Appro Facturation'].sum()

    # ── KPI J-1 depuis Excel ─────────────────────────────────
    df_j1 = pd.read_excel(
        fichier_excel,
        sheet_name="database_Eaux",
        header=0
    )
    df_j1.columns = df_j1.columns.str.strip()
    df_j1 = df_j1[df_j1['TCM'].notna()].copy()
    df_j1 = df_j1[
        pd.to_numeric(df_j1['TCM'], errors='coerce') >= 100
    ].copy()
    df_j1 = df_j1.sort_values('Date').reset_index(drop=True)

    derniere = df_j1.iloc[-1]
    COL_APPRO = 'E_Appro Facturation'

    if 'KPI EAU' in df_j1.columns:
        kpi_j1 = safe_float(derniere['KPI EAU'])
    else:
        kpi_j1 = np.nan

    if np.isnan(kpi_j1) if isinstance(kpi_j1, float) else False:
        kpi_j1 = safe_float(derniere[COL_APPRO]) / \
                 safe_float(derniere['TCM'])

    date_j1  = pd.Timestamp(derniere['Date']).date()
    tcm_j1   = safe_float(derniere['TCM'])
    conso_j1 = safe_float(derniere[COL_APPRO])

    # ── KPI PRÉDIT ANNÉE ─────────────────────────────────────
    fin_annee  = pd.Timestamp(f"{today.year}-12-31")
    jours_rest = pd.bdate_range(
        start=today + pd.Timedelta(days=1),
        end=fin_annee
    )
    tcm_moyen = df_prod['TCM'].mean()

    df_prophet_base = df_prod[['Date','KPI_m3_veh','TCM']].copy()
    df_prophet_base.columns = ['ds','y','TCM']
    df_prophet_base = df_prophet_base.dropna(subset=['y'])
    df_prophet_base = df_prophet_base[df_prophet_base['y'] < 3.0]

    df_future_ann = pd.concat([
        df_prophet_base[['ds','TCM']],
        pd.DataFrame({'ds': jours_rest, 'TCM': tcm_moyen})
    ], ignore_index=True)

    forecast_ann   = model_prophet.predict(df_future_ann)
    pred_restants  = forecast_ann[
        forecast_ann['ds'].isin(jours_rest)
    ]['yhat']
    pred_restants  = pred_restants.clip(lower=0.5, upper=2.0)
    kpi_predit_ann = pred_restants.mean()

    df_ytd         = df_kpi[df_kpi['Date'] >= debut_ann]
    nb_jours_reel  = len(df_ytd)
    nb_jours_futur = len(pred_restants)
    kpi_reel_moy   = df_ytd['KPI_m3_veh'].mean()
    poids_reel     = nb_jours_reel
    poids_futur    = nb_jours_futur * 0.3

    kpi_annee_complet = (
        (kpi_reel_moy * poids_reel + kpi_predit_ann * poids_futur)
        / (poids_reel + poids_futur)
    )

    # ── STATUT ───────────────────────────────────────────────
    OBJECTIF = 1.25

    def statut_kpi(val):
        if pd.isna(val):
            return {"emoji":"❓","texte":"Donnée invalide",
                    "couleur":"#666666","bg":"#E0E0E0"}
        if val > OBJECTIF * 1.15:
            return {"emoji":"🔴","texte":"Au-dessus objectif",
                    "couleur":"#A32D2D","bg":"#FCEBEB"}
        elif val > OBJECTIF:
            return {"emoji":"🟡","texte":"Proche objectif",
                    "couleur":"#854F0B","bg":"#FAEEDA"}
        else:
            return {"emoji":"🟢","texte":"Sous objectif ✓",
                    "couleur":"#27500A","bg":"#EAF3DE"}

    result = {
        'kpi_ytd'          : kpi_ytd,
        'kpi_mtd'          : kpi_mtd,
        'kpi_j1'           : kpi_j1,
        'date_j1'          : date_j1,
        'tcm_j1'           : tcm_j1,
        'conso_j1'         : conso_j1,
        'cumul_ytd'        : cumul_ytd,
        'kpi_predit_annee' : kpi_annee_complet,
        'nb_jours_reel'    : nb_jours_reel,
        'nb_jours_futur'   : nb_jours_futur,
        'objectif'         : OBJECTIF,
        'statut_ytd'       : statut_kpi(kpi_ytd),
        'statut_mtd'       : statut_kpi(kpi_mtd),
        'statut_j1'        : statut_kpi(kpi_j1),
        'statut_annee'     : statut_kpi(kpi_annee_complet),
    }

    print("\n=== KPIs DASHBOARD ===")
    print(f"Cumul YTD          : {cumul_ytd:.0f} m³")
    print(f"KPI YTD            : {kpi_ytd:.3f} m³/véh  {statut_kpi(kpi_ytd)['emoji']}")
    print(f"KPI MTD            : {kpi_mtd:.3f} m³/véh  {statut_kpi(kpi_mtd)['emoji']}")
    print(f"KPI J-1 ({date_j1}) : {kpi_j1:.3f} m³/véh  {statut_kpi(kpi_j1)['emoji']}")
    print(f"KPI Prédit 2026    : {kpi_annee_complet:.3f} m³/véh  {statut_kpi(kpi_annee_complet)['emoji']}")
    print(f"Objectif 2026      : {OBJECTIF} m³/véh")

    return result

kpis = calculer_kpis_dashboard(df_prod, model_prophet)

KPI YTD : 155040 / 121276 = 1.278 ✅
KPI MTD : 27303 / 21060 = 1.296 (mai) ✅

=== KPIs DASHBOARD ===
Cumul YTD          : 107458 m³
KPI YTD            : 1.278 m³/véh  🟡
KPI MTD            : 1.296 m³/véh  🟡
KPI J-1 (2026-05-20) : 1.189 m³/véh  🟢
KPI Prédit 2026    : 1.278 m³/véh  🟡
Objectif 2026      : 1.25 m³/véh


In [13]:
def clean_val(v):
    if v is None:
        return 0
    try:
        f = float(v)
        return 0 if (np.isnan(f) or np.isinf(f)) else f
    except:
        return 0

try:
    worksheet_kpi = sheet.worksheet("KPIs_Dashboard")
except gspread.exceptions.WorksheetNotFound:
    worksheet_kpi = sheet.add_worksheet(
        title="KPIs_Dashboard", rows=10, cols=12
    )

kpis_export = pd.DataFrame([{
    'date_calcul'      : str(date.today()),
    'kpi_ytd'          : round(clean_val(kpis['kpi_ytd']), 3),
    'kpi_mtd'          : round(clean_val(kpis['kpi_mtd']), 3),
    'kpi_j1'           : round(clean_val(kpis['kpi_j1']), 3),
    'date_j1'          : str(kpis['date_j1']),
    'tcm_j1'           : int(clean_val(kpis['tcm_j1'])),
    'conso_j1'         : round(clean_val(kpis['conso_j1']), 0),
    'cumul_ytd'        : round(clean_val(kpis['cumul_ytd']), 0),
    'kpi_predit_annee' : round(clean_val(kpis['kpi_predit_annee']), 3),
    'objectif'         : 1.25,
    'nb_jours_reel'    : int(clean_val(kpis['nb_jours_reel'])),
    'nb_jours_futur'   : int(clean_val(kpis['nb_jours_futur']))
}])

kpis_export = kpis_export.replace([np.inf, -np.inf], 0).fillna(0)

data_kpi = [kpis_export.columns.tolist()]
for _, row in kpis_export.iterrows():
    ligne = []
    for val in row.values:
        if isinstance(val, np.integer):
            ligne.append(int(val))
        elif isinstance(val, np.floating):
            ligne.append(float(val))
        else:
            ligne.append(str(val))
    data_kpi.append(ligne)

worksheet_kpi.clear()
worksheet_kpi.update(data_kpi)

print("=== KPIs exportés ✓ ===")
print(f"  YTD    : {kpis['kpi_ytd']:.3f} m³/véh")
print(f"  MTD    : {kpis['kpi_mtd']:.3f} m³/véh")
print(f"  J-1    : {kpis['kpi_j1']:.3f} m³/véh")
print(f"  Prédit : {kpis['kpi_predit_annee']:.3f} m³/véh")
print(f"  Cumul  : {kpis['cumul_ytd']:.0f} m³")

=== KPIs exportés ✓ ===
  YTD    : 1.278 m³/véh
  MTD    : 1.296 m³/véh
  J-1    : 1.189 m³/véh
  Prédit : 1.278 m³/véh
  Cumul  : 107458 m³


In [14]:
import pandas as pd
import numpy as np
import gspread
import time

fichier_excel = "../data/Synthèse_Eaux_2026_VF_(2).xlsm"

print("[1/3] Lecture KPIs officiels Synthèse_mois...")
df_mois = pd.read_excel(
    fichier_excel,
    sheet_name='Synthèse_mois',
    header=None
)

# KPI YTD
ligne_total = df_mois.iloc[16]
tcm_total   = safe_float(ligne_total[1])
appro_total = safe_float(ligne_total[2])
kpi_ytd_officiel = round(appro_total / tcm_total, 3) \
                   if tcm_total and tcm_total > 0 else 0

# KPI MTD par mois
kpi_par_mois = {}
for idx in range(4, 16):
    ligne   = df_mois.iloc[idx]
    tcm_m   = safe_float(ligne[1])
    appro_m = safe_float(ligne[2])
    nom     = str(ligne[0]).strip()
    mois_num = idx - 3
    if tcm_m and appro_m and tcm_m > 0:
        kpi_par_mois[mois_num] = round(appro_m / tcm_m, 3)
        print(f"  {nom:<6} (mois {mois_num}) : {kpi_par_mois[mois_num]:.3f}")

print(f"KPI YTD officiel : {kpi_ytd_officiel:.3f}")

print("\n[2/3] Construction historique journalier...")
df_db = pd.read_excel(
    fichier_excel,
    sheet_name='database_Eaux',
    header=0
)
df_db.columns = df_db.columns.str.strip()
df_db = df_db[df_db['TCM'].notna()].copy()
df_db = df_db[
    pd.to_numeric(df_db['TCM'], errors='coerce') > 0
].copy()
df_db['Date'] = pd.to_datetime(df_db['Date'], errors='coerce')
df_db = df_db[df_db['Date'].notna()].copy()
df_db = df_db.sort_values('Date').reset_index(drop=True)

# Rang : 1 = dernier jour
df_db['Rang'] = range(len(df_db), 0, -1)

# KPI_J1 depuis colonne KPI EAU
df_db['KPI_J1'] = pd.to_numeric(
    df_db['KPI EAU'], errors='coerce'
).round(3)

# KPI YTD et MTD officiels
df_db['KPI_YTD'] = kpi_ytd_officiel
df_db['mois_num'] = df_db['Date'].dt.month.astype(int)
df_db['KPI_MTD']  = df_db['mois_num'].map(kpi_par_mois).fillna(0).round(3)
df_db = df_db.drop(columns=['mois_num'])

# Colonnes à exporter
cols_base = [
    'Date', 'TCM', 'KPI_J1', 'KPI_YTD', 'KPI_MTD', 'Rang',
    'E_Appro Facturation', 'EI_Facturation', 'EP_Facturation',
    'ED_Total', 'EOR_Total'
]
cols_ei = [
    'EI_Peinture', 'EI_STEP', 'EI_AEB', 'EI_TC',
    'EI_TARs', 'EI _TOUR UE', 'EI_BD1&2',
    'EI _ST', 'EI_Bouclier', 'EI_Incendie',
    'EI Autres Process'
]
cols_ep = [
    'EP Cantines', 'EP Sanitaires', 'EP IFMIA',
    'EP BD NORD', 'EP BD SUD', 'EP ZC',
    'EP  U3', 'EP autres Cantines*', 'EP Arrosage'
]

cols_finales = (
    [c for c in cols_base if c in df_db.columns] +
    [c for c in cols_ei   if c in df_db.columns] +
    [c for c in cols_ep   if c in df_db.columns]
)

df_export = df_db[cols_finales].copy()

# Renommer colonnes
import re
def nettoyer_nom(col):
    col = col.strip().replace('*','').replace('&','et')
    return re.sub(r'\s+', '_', col)

df_export.columns = [nettoyer_nom(c) for c in df_export.columns]
df_export['Date'] = df_export['Date'].astype(str)

# Convertir valeurs
for col in df_export.columns:
    if col == 'Date':
        continue
    df_export[col] = df_export[col].apply(
        lambda x: round(float(x), 3)
        if pd.notna(x) and str(x) not in ['nan','inf','-inf','']
        else 0.0
    )

df_export = df_export.replace([np.inf, -np.inf], 0.0).fillna(0.0)

print(f"\n  Lignes  : {len(df_export)}")
print(f"  Période : {df_export['Date'].iloc[0]} → {df_export['Date'].iloc[-1]}")
print("\n=== VÉRIFICATION DERNIÈRE LIGNE ===")
d = df_export.iloc[-1]
print(f"  Date    : {d['Date']}")
print(f"  KPI_J1  : {d['KPI_J1']:.3f}")
print(f"  KPI_YTD : {d['KPI_YTD']:.3f}")
print(f"  KPI_MTD : {d['KPI_MTD']:.3f}")
print(f"  Rang    : {d['Rang']:.0f} ← doit être 1")

print("\n[3/3] Export Google Sheets...")
try:
    ws = sheet.worksheet("Historique_J1")
except gspread.exceptions.WorksheetNotFound:
    ws = sheet.add_worksheet(
        title="Historique_J1",
        rows=500,
        cols=len(df_export.columns) + 2
    )

time.sleep(1)
ws.clear()
time.sleep(1)

data = [df_export.columns.tolist()]
for _, row in df_export.iterrows():
    ligne = []
    for col in df_export.columns:
        val = row[col]
        if col == 'Date':
            ligne.append(str(val))
        else:
            try:
                f = float(val)
                ligne.append(0.0 if (np.isnan(f) or np.isinf(f)) else round(f,3))
            except:
                ligne.append(0.0)
    data.append(ligne)

chunk_size = 50
nb_blocs = (len(data) // chunk_size) + 1
for i in range(0, len(data), chunk_size):
    chunk = data[i: i + chunk_size]
    bloc_num = i // chunk_size + 1
    for attempt in range(3):
        try:
            ws.update(chunk, range_name=f"A{i+1}")
            print(f"  Bloc {bloc_num}/{nb_blocs} ✓")
            time.sleep(0.8)
            break
        except Exception as e:
            print(f"  Retry {attempt+1}/3 : {e}")
            time.sleep(5)

print()
print("=" * 50)
print("  HISTORIQUE_J1 EXPORTÉ ✓")
print("=" * 50)
print(f"  Lignes   : {len(df_export)}")
print(f"  Colonnes : {len(df_export.columns)}")

[1/3] Lecture KPIs officiels Synthèse_mois...
  janv   (mois 1) : 1.447
  févr   (mois 2) : 1.107
  mars   (mois 3) : 1.261
  avr    (mois 4) : 1.333
  mai    (mois 5) : 1.296
KPI YTD officiel : 1.278

[2/3] Construction historique journalier...

  Lignes  : 113
  Période : 2026-01-05 → 2026-05-20

=== VÉRIFICATION DERNIÈRE LIGNE ===
  Date    : 2026-05-20
  KPI_J1  : 1.189
  KPI_YTD : 1.278
  KPI_MTD : 1.296
  Rang    : 1 ← doit être 1

[3/3] Export Google Sheets...
  Bloc 1/3 ✓
  Bloc 2/3 ✓
  Bloc 3/3 ✓

  HISTORIQUE_J1 EXPORTÉ ✓
  Lignes   : 113
  Colonnes : 26


In [31]:
# ── EXPORT PEINTURE_J1 — Page 2 Dashboard ────────────────────
import pandas as pd
import numpy as np
import gspread
import time
import re

fichier_excel = "../data/Synthèse_Eaux_2026_VF_(2).xlsm"

print("[1/2] Lecture database_Eaux pour Peinture...")

df_db = pd.read_excel(
    fichier_excel,
    sheet_name='database_Eaux',
    header=0
)
df_db.columns = df_db.columns.str.strip()
df_db = df_db[df_db['TCM'].notna()].copy()
df_db = df_db[
    pd.to_numeric(df_db['TCM'], errors='coerce') > 0
].copy()
df_db['Date'] = pd.to_datetime(df_db['Date'], errors='coerce')
df_db = df_db[df_db['Date'].notna()].copy()
df_db = df_db.sort_values('Date').reset_index(drop=True)

# ── COLONNES PEINTURE ────────────────────────────────────────
# Disque GAUCHE : EI_TTS CATA_PA + EI_Application PA (col Y, Z)
# Disque CENTRE : EI_Peinture + ED_Peinture (col AA, AD)
# Disque DROITE : ED_TTS CATA_PA + ED_Application PA (col AB, AC)

cols_peinture = [
    'Date', 'TCM',
    'EI_Peinture',        # disque centre
    'ED_Peinture',        # disque centre
    'EI_TTS CATA_PA',     # disque gauche
    'EI_Application PA',  # disque gauche
    'ED_TTS CATA_PA',     # disque droite
    'ED_Application PA',  # disque droite
]

# Garder uniquement colonnes disponibles
cols_dispo = [c for c in cols_peinture if c in df_db.columns]
print(f"  Colonnes trouvées : {cols_dispo}")

df_peinture = df_db[cols_dispo].copy()

# Nettoyage
def nettoyer_nom(col):
    col = col.strip().replace('*','').replace('&','et')
    return re.sub(r'\s+', '_', col)

df_peinture.columns = [nettoyer_nom(c) for c in df_peinture.columns]
df_peinture['Date'] = df_peinture['Date'].astype(str)

for col in df_peinture.columns:
    if col == 'Date':
        continue
    df_peinture[col] = df_peinture[col].apply(
        lambda x: round(float(x), 3)
        if pd.notna(x) and str(x) not in ['nan','inf','-inf','']
        else 0.0
    )
df_peinture = df_peinture.replace([np.inf, -np.inf], 0.0).fillna(0.0)

print(f"  Lignes    : {len(df_peinture)}")
print(f"  Période   : {df_peinture['Date'].iloc[0]} → {df_peinture['Date'].iloc[-1]}")
print()
print("=== VÉRIFICATION DERNIÈRE LIGNE ===")
d = df_peinture.iloc[-1]
print(f"  Date              : {d['Date']}")
print(f"  EI_Peinture       : {d['EI_Peinture']:.0f} m³  (disque centre)")
print(f"  ED_Peinture       : {d['ED_Peinture']:.0f} m³  (disque centre)")
print(f"  EI_TTS_CATA_PA    : {d['EI_TTS_CATA_PA']:.0f} m³  (disque gauche)")
print(f"  EI_Application_PA : {d['EI_Application_PA']:.0f} m³  (disque gauche)")
print(f"  ED_TTS_CATA_PA    : {d['ED_TTS_CATA_PA']:.0f} m³  (disque droite)")
print(f"  ED_Application_PA : {d['ED_Application_PA']:.0f} m³  (disque droite)")

# ── EXPORT VERS GOOGLE SHEETS ─────────────────────────────────
print("\n[2/2] Export Google Sheets...")

try:
    ws_p = sheet.worksheet("Peinture_J1")
    print("  Onglet Peinture_J1 trouvé ✓")
except gspread.exceptions.WorksheetNotFound:
    ws_p = sheet.add_worksheet(
        title="Peinture_J1",
        rows=500,
        cols=len(df_peinture.columns) + 2
    )
    print("  Onglet Peinture_J1 créé ✓")

time.sleep(1)
ws_p.clear()
time.sleep(1)

# Convertir en types Python natifs
data = [df_peinture.columns.tolist()]
for _, row in df_peinture.iterrows():
    ligne = []
    for col in df_peinture.columns:
        val = row[col]
        if col == 'Date':
            ligne.append(str(val))
        else:
            try:
                f = float(val)
                ligne.append(
                    0.0 if (np.isnan(f) or np.isinf(f))
                    else round(f, 3)
                )
            except:
                ligne.append(0.0)
    data.append(ligne)

# Envoi par blocs
chunk_size = 50
nb_blocs = (len(data) // chunk_size) + 1
for i in range(0, len(data), chunk_size):
    chunk    = data[i: i + chunk_size]
    bloc_num = i // chunk_size + 1
    for attempt in range(3):
        try:
            ws_p.update(chunk, range_name=f"A{i+1}")
            print(f"  Bloc {bloc_num}/{nb_blocs} ✓")
            time.sleep(0.8)
            break
        except Exception as e:
            print(f"  Retry {attempt+1}/3 : {e}")
            time.sleep(5)

print()
print("=" * 55)
print("  PEINTURE_J1 EXPORTÉ ✓")
print("=" * 55)
print()


[1/2] Lecture database_Eaux pour Peinture...
  Colonnes trouvées : ['Date', 'TCM', 'EI_Peinture', 'ED_Peinture', 'EI_TTS CATA_PA', 'EI_Application PA', 'ED_TTS CATA_PA', 'ED_Application PA']
  Lignes    : 113
  Période   : 2026-01-05 → 2026-05-20

=== VÉRIFICATION DERNIÈRE LIGNE ===
  Date              : 2026-05-20
  EI_Peinture       : 378 m³  (disque centre)
  ED_Peinture       : 541 m³  (disque centre)
  EI_TTS_CATA_PA    : 313 m³  (disque gauche)
  EI_Application_PA : 65 m³  (disque gauche)
  ED_TTS_CATA_PA    : 477 m³  (disque droite)
  ED_Application_PA : 64 m³  (disque droite)

[2/2] Export Google Sheets...
  Onglet Peinture_J1 trouvé ✓
  Bloc 1/3 ✓
  Bloc 2/3 ✓
  Bloc 3/3 ✓

  PEINTURE_J1 EXPORTÉ ✓



In [33]:
# ── EXPORT 3 ONGLETS SÉPARÉS POUR LES DISQUES ───────────────

def creer_onglet_disque(sheet, nom, data_dict, df_dates):
    """
    Crée un onglet avec 3 colonnes :
    Date | Type | Valeur_m3
    Une ligne par type par jour
    """
    rows = []
    for _, row_db in df_dates.iterrows():
        date_str = str(row_db['Date'])
        for type_nom, col_nom in data_dict.items():
            val = row_db.get(col_nom, 0)
            try:
                val = float(val)
            except:
                val = 0.0
            rows.append({
                'Date'      : date_str,
                'Type'      : type_nom,
                'Valeur_m3' : round(val, 0)
            })

    df_out = pd.DataFrame(rows)

    try:
        ws = sheet.worksheet(nom)
    except gspread.exceptions.WorksheetNotFound:
        ws = sheet.add_worksheet(
            title=nom, rows=1000, cols=5
        )

    time.sleep(1)
    ws.clear()
    time.sleep(1)

    data = [df_out.columns.tolist()]
    for _, row in df_out.iterrows():
        data.append([
            str(row['Date']),
            str(row['Type']),
            float(row['Valeur_m3'])
        ])

    chunk_size = 100
    for i in range(0, len(data), chunk_size):
        chunk = data[i: i + chunk_size]
        for attempt in range(3):
            try:
                ws.update(chunk, range_name=f"A{i+1}")
                time.sleep(0.5)
                break
            except Exception as e:
                time.sleep(5)

    print(f"  '{nom}' exporté ✓ ({len(df_out)} lignes)")
    return ws

# Disque GAUCHE : EI TTS CATA + EI Application
creer_onglet_disque(
    sheet,
    "Peinture_Gauche",
    {
        'EI TTS CATA'    : 'EI_TTS_CATA_PA',
        'EI Application' : 'EI_Application_PA'
    },
    df_peinture
)

# Disque CENTRE : EI Peinture + ED Peinture
creer_onglet_disque(
    sheet,
    "Peinture_Centre",
    {
        'EI Peinture' : 'EI_Peinture',
        'ED Peinture' : 'ED_Peinture'
    },
    df_peinture
)

# Disque DROITE : ED TTS CATA + ED Application
creer_onglet_disque(
    sheet,
    "Peinture_Droite",
    {
        'ED TTS CATA'    : 'ED_TTS_CATA_PA',
        'ED Application' : 'ED_Application_PA'
    },
    df_peinture
)

print()
print("=== 3 ONGLETS DISQUES EXPORTÉS ✓ ===")
print("  Peinture_Gauche → EI TTS CATA + EI Application")
print("  Peinture_Centre → EI Peinture + ED Peinture")
print("  Peinture_Droite → ED TTS CATA + ED Application")
print()
print("  Structure de chaque onglet :")
print("  Date       | Type           | Valeur_m3")
print("  2026-01-01 | EI TTS CATA    | 116")
print("  2026-01-01 | EI Application | 26")
print("  2026-01-02 | EI TTS CATA    | 120")
print("  ...")
print()


  'Peinture_Gauche' exporté ✓ (226 lignes)
  'Peinture_Centre' exporté ✓ (226 lignes)
  'Peinture_Droite' exporté ✓ (226 lignes)

=== 3 ONGLETS DISQUES EXPORTÉS ✓ ===
  Peinture_Gauche → EI TTS CATA + EI Application
  Peinture_Centre → EI Peinture + ED Peinture
  Peinture_Droite → ED TTS CATA + ED Application

  Structure de chaque onglet :
  Date       | Type           | Valeur_m3
  2026-01-01 | EI TTS CATA    | 116
  2026-01-01 | EI Application | 26
  2026-01-02 | EI TTS CATA    | 120
  ...



In [34]:
def analyser_jour_courant(df_xgb, shap_values, model_xgb):
    """
    Analyse le dernier jour disponible dans le dataset.
    Retourne un dictionnaire avec toutes les infos du rapport.
    """
    idx      = len(df_xgb) - 1
    row      = df_xgb.iloc[idx]
    date_j   = row['Date'].date()
    kpi_reel = row['KPI_m3_veh']
    kpi_pred = model_xgb.predict(X.iloc[[idx]])[0]

    # Statut
    if kpi_reel > SEUIL_ALERTE:
        statut       = "ANOMALIE"
        statut_emoji = "🔴"
        couleur      = "#FCEBEB"
    elif kpi_reel > SEUIL_OBJECTIF:
        statut       = "ATTENTION"
        statut_emoji = "🟡"
        couleur      = "#FAEEDA"
    else:
        statut       = "NORMAL"
        statut_emoji = "🟢"
        couleur      = "#EAF3DE"

    # Top 3 causes SHAP — uniquement features métier
    contribs = pd.Series(shap_values[idx], index=FEATURES)

    # ── CORRECTION : exclure les features non parlantes ──────
    FEATURES_EXCLURE = [
        'jour_semaine', 'mois', 'trimestre',
        'is_lundi', 'taux_EP', 'taux_EI'
    ]
    contribs_metier = contribs.drop(
        [f for f in FEATURES_EXCLURE if f in contribs.index]
    )
    top3 = contribs_metier.abs().nlargest(3)
    # ─────────────────────────────────────────────────────────

    noms_lisibles = {
        'TCM'          : 'Production (TCM)',
        'ED_Total'     : 'Eau déminéralisée',
        'EOR_Total'    : 'Eau osmosée recyclée',
        'EI_Facturation': 'Eau industrielle (facturation)',
        'EP_Facturation': 'Eau potable (facturation)',
        'ratio_EOR_ED' : 'Taux recyclage EOR/ED',
    }

    causes = []
    for feat, _ in top3.items():
        val_shap   = contribs[feat]
        val_reelle = X.iloc[idx][feat]
        nom        = noms_lisibles.get(feat, feat)
        direction  = "↑ augmente" if val_shap > 0 else "↓ réduit"
        causes.append({
            'feature'  : nom,
            'valeur'   : val_reelle,
            'shap'     : val_shap,
            'direction': direction
        })

    return {
        'date'        : date_j,
        'kpi_reel'    : kpi_reel,
        'kpi_pred'    : kpi_pred,
        'tcm'         : row['TCM'],
        'conso'       : row['E_Appro Facturation'],
        'statut'      : statut,
        'statut_emoji': statut_emoji,
        'couleur'     : couleur,
        'causes'      : causes
    }

analyse = analyser_jour_courant(df_xgb, shap_values, model_xgb)
print(f"Date analysée  : {analyse['date']}")
print(f"Statut         : {analyse['statut_emoji']} {analyse['statut']}")
print(f"KPI réel       : {analyse['kpi_reel']:.3f} m³/véh")
print(f"KPI prédit     : {analyse['kpi_pred']:.3f} m³/véh")
print(f"TCM            : {analyse['tcm']:.0f} véhicules")
print()
print("Top 3 causes SHAP (features métier uniquement) :")
for c in analyse['causes']:
    print(f"  {c['direction']} le KPI — {c['feature']} = {c['valeur']:.1f}  (SHAP={c['shap']:+.3f})")

Date analysée  : 2026-05-20
Statut         : 🟢 NORMAL
KPI réel       : 1.184 m³/véh
KPI prédit     : 1.096 m³/véh
TCM            : 1410 véhicules

Top 3 causes SHAP (features métier uniquement) :
  ↓ réduit le KPI — Production (TCM) = 1410.0  (SHAP=-0.109)
  ↑ augmente le KPI — Eau industrielle (facturation) = 1181.0  (SHAP=+0.107)
  ↑ augmente le KPI — Eau potable (facturation) = 447.0  (SHAP=+0.039)


In [35]:
def predire_7_jours(model_prophet, df_prod):
    """
    Génère les prédictions à partir d'AUJOURD'HUI
    et non depuis la dernière date du dataset.
    """
    from datetime import date
    import pandas as pd

    tcm_moyen = df_prod['TCM'].mean()

    df_prophet = df_prod[['Date','KPI_m3_veh','TCM']].copy()
    df_prophet.columns = ['ds','y','TCM']
    df_prophet = df_prophet.dropna(subset=['y'])

    # ── CORRECTION : calculer combien de jours ouvrés
    # manquent entre la dernière date et aujourd'hui + 7 jours
    derniere_date = df_prophet['ds'].max()
    aujourd_hui   = pd.Timestamp(date.today())

    # Générer tous les jours ouvrés depuis la dernière date jusqu'à +7j futurs
    toutes_dates = pd.bdate_range(
        start=derniere_date + pd.Timedelta(days=1),
        end=aujourd_hui + pd.Timedelta(days=10)  # marge suffisante
    )

    # Garder uniquement les 7 prochains jours ouvrés FUTURS
    jours_futurs = pd.DataFrame({'ds': toutes_dates})
    jours_futurs = jours_futurs[jours_futurs['ds'] > aujourd_hui].head(7)

    # Construire le dataframe future complet pour Prophet
    df_future = pd.concat([
        df_prophet[['ds','TCM']],
        jours_futurs.assign(TCM=tcm_moyen)
    ], ignore_index=True)

    # Prédire
    forecast = model_prophet.predict(df_future)

    # Garder uniquement les 7 jours futurs
    predictions = forecast[forecast['ds'].isin(jours_futurs['ds'])][
        ['ds','yhat','yhat_lower','yhat_upper']
    ].copy()
    predictions.columns = ['date','kpi_predit','borne_basse','borne_haute']
    predictions['statut'] = predictions['kpi_predit'].apply(
        lambda x: '🔴 Alerte'    if x > SEUIL_ALERTE
                  else ('🟡 Attention' if x > SEUIL_OBJECTIF
                  else '🟢 Normal')
    )

    return predictions

predictions_7j = predire_7_jours(model_prophet, df_prod)

print("=== PRÉDICTIONS 7 PROCHAINS JOURS RÉELS ===")
print(predictions_7j.round(3).to_string(index=False))

=== PRÉDICTIONS 7 PROCHAINS JOURS RÉELS ===
      date  kpi_predit  borne_basse  borne_haute      statut
2026-05-22       1.285        1.118        1.477 🟡 Attention
2026-05-25       1.257        1.066        1.441 🟡 Attention
2026-05-26       1.295        1.113        1.490 🟡 Attention
2026-05-27       1.373        1.191        1.550 🟡 Attention
2026-05-28       1.341        1.154        1.518 🟡 Attention
2026-05-29       1.314        1.137        1.517 🟡 Attention


In [36]:
from datetime import date
import pandas as pd
import numpy as np

# ── EXPORTER PRÉDICTIONS (onglet 2) ─────────────────────────
try:
    worksheet_pred = sheet.worksheet("Predictions")
    print("Onglet Predictions trouvé ✓")
except gspread.exceptions.WorksheetNotFound:
    worksheet_pred = sheet.add_worksheet(
        title="Predictions", rows=20, cols=6
    )
    print("Onglet Predictions créé ✓")

pred_export = predictions_7j.copy()
pred_export['date'] = pred_export['date'].astype(str)
pred_export = pred_export.replace([np.inf, -np.inf], 0)
pred_export = pred_export.fillna(0)
pred_export = pred_export.round(3)

data_pred = [pred_export.columns.tolist()] + pred_export.values.tolist()
worksheet_pred.clear()
worksheet_pred.update(data_pred)
print(f"Prédictions exportées : {len(pred_export)} lignes ✓")

# ── EXPORTER KPIs DASHBOARD (onglet 3) ──────────────────────
try:
    worksheet_kpi = sheet.worksheet("KPIs_Dashboard")
    print("Onglet KPIs_Dashboard trouvé ✓")
except gspread.exceptions.WorksheetNotFound:
    worksheet_kpi = sheet.add_worksheet(
        title="KPIs_Dashboard", rows=10, cols=12
    )
    print("Onglet KPIs_Dashboard créé ✓")

# Fonction pour nettoyer une valeur
def clean_val(v):
    if v is None:
        return 0
    if isinstance(v, float) and (np.isnan(v) or np.isinf(v)):
        return 0
    return v

kpis_export = pd.DataFrame([{
    'date_calcul'      : str(date.today()),
    'kpi_ytd'          : round(clean_val(kpis['kpi_ytd']), 3),
    'kpi_mtd'          : round(clean_val(kpis['kpi_mtd']), 3),
    'kpi_j1'           : round(clean_val(kpis['kpi_j1']), 3),
    'date_j1'          : str(kpis['date_j1']),
    'cumul_ytd'        : round(clean_val(kpis['cumul_ytd']), 0),
    'tcm_j1'           : int(clean_val(kpis['tcm_j1'])),
    'conso_j1'         : round(clean_val(kpis['conso_j1']), 0),
    'kpi_predit_annee' : round(clean_val(kpis['kpi_predit_annee']), 3),
    'objectif'         : 1.25,
    'nb_jours_reel'    : int(clean_val(kpis['nb_jours_reel'])),
    'nb_jours_futur'   : int(clean_val(kpis['nb_jours_futur']))
}])

# Nettoyer le dataframe complet
kpis_export = kpis_export.replace([np.inf, -np.inf], 0)
kpis_export = kpis_export.fillna(0)

# Convertir en liste Python native (pas numpy)
data_kpi = [kpis_export.columns.tolist()]
for _, row in kpis_export.iterrows():
    ligne = []
    for val in row.values:
        if isinstance(val, (np.integer)):
            ligne.append(int(val))
        elif isinstance(val, (np.floating)):
            ligne.append(float(val))
        else:
            ligne.append(val)
    data_kpi.append(ligne)

worksheet_kpi.clear()
worksheet_kpi.update(data_kpi)

print()
print("=== Export Google Sheets terminé ✓ ===")
print(f"  YTD    : {kpis['kpi_ytd']:.3f} m³/véh")
print(f"  MTD    : {kpis['kpi_mtd']:.3f} m³/véh")
print(f"  J-1    : {kpis['kpi_j1']:.3f} m³/véh")
print(f"  Prédit : {kpis['kpi_predit_annee']:.3f} m³/véh")
print(f"Volume J-1 : {kpis['conso_j1']:.0f} m³")
print(f"Date J-1   : {kpis['date_j1']}")
print()
print("3 onglets dans Google Sheets :")
print("  Feuille 1      → données historiques")
print("  Predictions    → prévisions 7 jours")
print("  KPIs_Dashboard → KPIs calculés Python")

Onglet Predictions trouvé ✓
Prédictions exportées : 6 lignes ✓
Onglet KPIs_Dashboard trouvé ✓

=== Export Google Sheets terminé ✓ ===
  YTD    : 1.278 m³/véh
  MTD    : 1.296 m³/véh
  J-1    : 1.189 m³/véh
  Prédit : 1.278 m³/véh
Volume J-1 : 1676 m³
Date J-1   : 2026-05-20

3 onglets dans Google Sheets :
  Feuille 1      → données historiques
  Predictions    → prévisions 7 jours
  KPIs_Dashboard → KPIs calculés Python


In [37]:
import time
import gspread

def export_avec_retry(worksheet, data, max_retries=3):
    """Envoie les données avec retry automatique en cas de coupure."""
    for attempt in range(max_retries):
        try:
            worksheet.clear()
            time.sleep(1)
            worksheet.update(data)
            return True
        except Exception as e:
            print(f"  Tentative {attempt + 1}/{max_retries} échouée : {e}")
            if attempt < max_retries - 1:
                time.sleep(5)
            else:
                print("  Echec après 3 tentatives.")
                raise e

# Créer ou ouvrir l'onglet J1_Data
try:
    worksheet_j1 = sheet.worksheet("J1_Data")
    print("Onglet J1_Data trouvé ✓")
except gspread.exceptions.WorksheetNotFound:
    worksheet_j1 = sheet.add_worksheet(
        title="J1_Data", rows=5, cols=4
    )
    print("Onglet J1_Data créé ✓")

time.sleep(2)  # pause avant lecture Excel

# Lire le fichier Excel
df_j1 = pd.read_excel(
    "../data/Synthèse_Eaux_2026_VF_(2).xlsm",
    sheet_name="database_Eaux",
    header=0
)
df_j1.columns = df_j1.columns.str.strip()
df_j1 = df_j1[df_j1['TCM'] >= 100].dropna(subset=['TCM'])
df_j1 = df_j1.sort_values('Date').reset_index(drop=True)
derniere = df_j1.iloc[-1]

# Identifier les colonnes EI et EP
col_ei = 'EI_Facturation' if 'EI_Facturation' in df_j1.columns else 'EI_Looker'
col_ep = 'EP_Facturation' if 'EP_Facturation' in df_j1.columns else 'EP_Looker'

val_ei = round(float(derniere[col_ei]), 0) if col_ei in derniere.index else 0
val_ep = round(float(derniere[col_ep]), 0) if col_ep in derniere.index else 0
date_j1_str = str(pd.Timestamp(derniere['Date']).date())

# Préparer les données
df_j1_export = pd.DataFrame([
    {'Type_Eau': 'EI Facturation', 'Volume_m3': val_ei, 'Date_J1': date_j1_str},
    {'Type_Eau': 'EP Facturation', 'Volume_m3': val_ep, 'Date_J1': date_j1_str}
])

data_j1 = [df_j1_export.columns.tolist()] + df_j1_export.values.tolist()

# Exporter avec retry
export_avec_retry(worksheet_j1, data_j1)

print(f"J1_Data exporté ✓")
print(f"  EI J-1 : {val_ei:.0f} m³  ({col_ei})")
print(f"  EP J-1 : {val_ep:.0f} m³  ({col_ep})")
print(f"  Date   : {date_j1_str}")

Onglet J1_Data trouvé ✓
J1_Data exporté ✓
  EI J-1 : 1181 m³  (EI_Facturation)
  EP J-1 : 447 m³  (EP_Facturation)
  Date   : 2026-05-20


In [38]:

# ── EXPORT DONNÉES ANNEAU EI/EP ──────────────────────────────
try:
    worksheet_anneau = sheet.worksheet("EI_EP_YTD")
except:
    worksheet_anneau = sheet.add_worksheet(
        title="EI_EP_YTD", rows=10, cols=4
    )

df = pd.read_csv(
    "../outputs/dataset_eau_propre.csv",
    parse_dates=['Date']
)

# Filtrer YTD jours production
from datetime import date
debut_ann = pd.Timestamp(f"{date.today().year}-01-01")
df_ytd = df[
    (df['Date'] >= debut_ann) &
    (df['TCM'] >= 100) &
    (df['is_weekend'] == 0)
].copy()

# Calculer totaux YTD
total_ei = df_ytd['EI_Facturation'].sum()
total_ep = df_ytd['EP_Facturation'].sum() \
           if 'EP_Facturation' in df_ytd.columns \
           else df_ytd['EP_Looker'].sum()
total    = total_ei + total_ep

# Créer le dataframe pour l'anneau
df_anneau = pd.DataFrame([
    {
        'Type_Eau'   : 'EI Facturation',
        'Volume_m3'  : round(total_ei, 0),
        'Pourcentage': round((total_ei / total) * 100, 1)
    },
    {
        'Type_Eau'   : 'EP Facturation',
        'Volume_m3'  : round(total_ep, 0),
        'Pourcentage': round((total_ep / total) * 100, 1)
    }
])

data_anneau = [df_anneau.columns.tolist()] + \
              df_anneau.values.tolist()
worksheet_anneau.clear()
worksheet_anneau.update(data_anneau)

print("=== EXPORT EI/EP ANNEAU ✓ ===")
print(f"EI Facturation : {total_ei:.0f} m³ ({total_ei/total*100:.1f}%)")
print(f"EP Facturation : {total_ep:.0f} m³ ({total_ep/total*100:.1f}%)")
print(f"Total          : {total:.0f} m³")

=== EXPORT EI/EP ANNEAU ✓ ===
EI Facturation : 75618 m³ (73.0%)
EP Facturation : 27988 m³ (27.0%)
Total          : 103606 m³


In [39]:
# ── EXPORT EI PROCESS YTD + EP PROCESS YTD ──────────────────

fichier_excel = "../data/Synthèse_Eaux_2026_VF_(2).xlsm"

# ── LECTURE EAU INDUSTRIELLE PAR PROCESS ────────────────────
df_ei_raw = pd.read_excel(
    fichier_excel,
    sheet_name='Synthèse Eau Industrielle',
    header=None
)

# Les données mensuelles sont en ligne 3 à 6 (janv→avr)
# Colonnes : 2=EI_Fact, 3=Peinture, 4=TARs, 5=STEP,
#            6=AEB, 7=TC, 8=Bouclier, 9=Incendie, 10=Autres

# Prendre les lignes avec données (janv à mois actuel)
# Row index 3=janv, 4=fev, 5=mars, 6=avr
df_ei_mois = df_ei_raw.iloc[3:7, [3, 4, 5, 6, 7, 8, 9, 10]].copy()
df_ei_mois.columns = [
    'EI_Peinture', 'EI_TARs', 'EI_STEP',
    'EI_AEB', 'EI_TC', 'EI_Bouclier',
    'EI_Incendie', 'EI_Autres'
]

# Convertir en numérique et sommer YTD
df_ei_mois = df_ei_mois.apply(pd.to_numeric, errors='coerce').fillna(0)
totaux_ei  = df_ei_mois.sum()

# Créer le dataframe pour le cercle
noms_ei = {
    'EI_Peinture'  : 'Peinture',
    'EI_TARs'      : 'TARs',
    'EI_STEP'      : 'STEP',
    'EI_AEB'       : 'AEB',
    'EI_TC'        : 'TC',
    'EI_Bouclier'  : 'Bouclier',
    'EI_Incendie'  : 'Incendie',
    'EI_Autres'    : 'Autres Process'
}

df_ei_process = pd.DataFrame([
    {
        'Process'   : noms_ei[col],
        'Volume_m3' : round(float(totaux_ei[col]), 0),
        'Type'      : 'EI'
    }
    for col in totaux_ei.index
    if totaux_ei[col] > 0
])

print("=== EI PAR PROCESS YTD ===")
total_ei_process = df_ei_process['Volume_m3'].sum()
for _, row in df_ei_process.iterrows():
    pct = row['Volume_m3'] / total_ei_process * 100
    print(f"  {row['Process']:<20} : {row['Volume_m3']:>8.0f} m³  ({pct:.1f}%)")
print(f"  {'TOTAL':<20} : {total_ei_process:>8.0f} m³")

# ── LECTURE EAU POTABLE PAR PROCESS ─────────────────────────
df_ep_raw = pd.read_excel(
    fichier_excel,
    sheet_name='Synthèse Eau Potable',
    header=None
)

# Colonnes : 3=Cantines, 4=Sanitaires, 5=IFMIA, 6=Arrosage
df_ep_mois = df_ep_raw.iloc[3:7, [3, 4, 5, 6]].copy()
df_ep_mois.columns = [
    'EP_Cantines', 'EP_Sanitaires',
    'EP_IFMIA', 'EP_Arrosage'
]

df_ep_mois = df_ep_mois.apply(pd.to_numeric, errors='coerce').fillna(0)
totaux_ep  = df_ep_mois.sum()

noms_ep = {
    'EP_Cantines'   : 'Cantines',
    'EP_Sanitaires' : 'Sanitaires',
    'EP_IFMIA'      : 'IFMIA',
    'EP_Arrosage'   : 'Arrosage'
}

df_ep_process = pd.DataFrame([
    {
        'Process'   : noms_ep[col],
        'Volume_m3' : round(float(totaux_ep[col]), 0),
        'Type'      : 'EP'
    }
    for col in totaux_ep.index
    if totaux_ep[col] >= 0
])

print()
print("=== EP PAR PROCESS YTD ===")
total_ep_process = df_ep_process['Volume_m3'].sum()
for _, row in df_ep_process.iterrows():
    pct = row['Volume_m3'] / total_ep_process * 100 if total_ep_process > 0 else 0
    print(f"  {row['Process']:<20} : {row['Volume_m3']:>8.0f} m³  ({pct:.1f}%)")
print(f"  {'TOTAL':<20} : {total_ep_process:>8.0f} m³")

=== EI PAR PROCESS YTD ===
  Peinture             :    36807 m³  (38.9%)
  TARs                 :     7072 m³  (7.5%)
  STEP                 :    16151 m³  (17.1%)
  AEB                  :     7427 m³  (7.9%)
  TC                   :     2592 m³  (2.7%)
  Bouclier             :      412 m³  (0.4%)
  Incendie             :     5420 m³  (5.7%)
  Autres Process       :    18703 m³  (19.8%)
  TOTAL                :    94584 m³

=== EP PAR PROCESS YTD ===
  Cantines             :     7469 m³  (25.5%)
  Sanitaires           :    21531 m³  (73.5%)
  IFMIA                :      147 m³  (0.5%)
  Arrosage             :      135 m³  (0.5%)
  TOTAL                :    29282 m³


In [40]:
# ── EXPORT EI PROCESS → Google Sheets ───────────────────────
try:
    ws_ei = sheet.worksheet("EI_Process_YTD")
    print("Onglet EI_Process_YTD trouvé ✓")
except gspread.exceptions.WorksheetNotFound:
    ws_ei = sheet.add_worksheet(
        title="EI_Process_YTD", rows=15, cols=4
    )
    print("Onglet EI_Process_YTD créé ✓")

import time
time.sleep(1)

data_ei = [df_ei_process.columns.tolist()] + \
          df_ei_process.values.tolist()
ws_ei.clear()
ws_ei.update(data_ei)
print(f"EI_Process_YTD exporté : {len(df_ei_process)} process ✓")

time.sleep(2)

# ── EXPORT EP PROCESS → Google Sheets ───────────────────────
try:
    ws_ep = sheet.worksheet("EP_Process_YTD")
    print("Onglet EP_Process_YTD trouvé ✓")
except gspread.exceptions.WorksheetNotFound:
    ws_ep = sheet.add_worksheet(
        title="EP_Process_YTD", rows=10, cols=4
    )
    print("Onglet EP_Process_YTD créé ✓")

time.sleep(1)

data_ep = [df_ep_process.columns.tolist()] + \
          df_ep_process.values.tolist()
ws_ep.clear()
ws_ep.update(data_ep)
print(f"EP_Process_YTD exporté : {len(df_ep_process)} process ✓")
print()
print("Google Sheets contient maintenant 5 onglets :")
print("  Feuille 1       → données historiques")
print("  KPIs_Dashboard  → KPIs calculés")
print("  Predictions     → prévisions 7 jours")
print("  J1_Data         → EI/EP J-1")
print("  EI_Process_YTD  → EI par process YTD")
print("  EP_Process_YTD  → EP par process YTD")

Onglet EI_Process_YTD trouvé ✓
EI_Process_YTD exporté : 8 process ✓
Onglet EP_Process_YTD trouvé ✓
EP_Process_YTD exporté : 4 process ✓

Google Sheets contient maintenant 5 onglets :
  Feuille 1       → données historiques
  KPIs_Dashboard  → KPIs calculés
  Predictions     → prévisions 7 jours
  J1_Data         → EI/EP J-1
  EI_Process_YTD  → EI par process YTD
  EP_Process_YTD  → EP par process YTD


In [41]:
# ── EXPORT EI PROCESS J-1 + EP PROCESS J-1 ──────────────────

fichier_excel = "../data/Synthèse_Eaux_2026_VF_(2).xlsm"

# Lire la feuille data J-1
df_j1_raw = pd.read_excel(
    fichier_excel,
    sheet_name='data J-1',
    header=0
)
df_j1_raw.columns = df_j1_raw.columns.str.strip()

# Prendre la première ligne (données J-1 réelles)
j1 = df_j1_raw.iloc[0]
date_j1_str = str(pd.Timestamp(j1['Date']).date())

print(f"=== DONNÉES J-1 : {date_j1_str} ===")

# ── EI PAR PROCESS J-1 ───────────────────────────────────────
ei_process_j1 = {
    'Peinture'      : float(j1.get('EI_Peinture', 0) or 0),
    'STEP'          : float(j1.get('EI_STEP', 0) or 0),
    'AEB'           : float(j1.get('EI_AEB', 0) or 0),
    'TC'            : float(j1.get('EI_TC', 0) or 0),
    'Des TAR'       : float(j1.get('EI_Des TAR', 0) or 0),
    'TOUR UE'       : float(j1.get('EI _TOUR UE', 0) or 0),
    'BD1&2'         : float(j1.get('EI_BD1&2', 0) or 0),
    'ST'            : float(j1.get('EI _ST', 0) or 0),
    'Bouclier'      : float(j1.get('EI_Bouclier', 0) or 0),
    'Incendie'      : float(j1.get('EI_Incendie', 0) or 0),
    'Autres Process': float(j1.get('EI Autres Process', 0) or 0),
}

df_ei_j1 = pd.DataFrame([
    {
        'Process'   : nom,
        'Volume_m3' : round(val, 0),
        'Date_J1'   : date_j1_str
    }
    for nom, val in ei_process_j1.items()
    if val > 0
])

print("EI par process J-1 :")
total_ei_j1 = df_ei_j1['Volume_m3'].sum()
for _, row in df_ei_j1.iterrows():
    pct = row['Volume_m3'] / total_ei_j1 * 100
    print(f"  {row['Process']:<20} : {row['Volume_m3']:>7.0f} m³ ({pct:.1f}%)")
print(f"  {'TOTAL':<20} : {total_ei_j1:>7.0f} m³")

# ── EP PAR PROCESS J-1 ───────────────────────────────────────
ep_process_j1 = {
    'Sanitaires'       : float(j1.get('EP Sanitaires', 0) or 0),
    'Cantines'         : float(j1.get('EP Cantines', 0) or 0),
    'Autres Cantines'  : float(j1.get('EP autres Cantines*', 0) or 0),
    'U3'               : float(j1.get('EP  U3', 0) or 0),
    'ZC'               : float(j1.get('EP ZC', 0) or 0),
    'BD SUD'           : float(j1.get('EP BD SUD', 0) or 0),
    'BD NORD'          : float(j1.get('EP BD NORD', 0) or 0),
    'IFMIA'            : float(j1.get('EP IFMIA', 0) or 0),
    'Arrosage'         : float(j1.get('EP Arrosage', 0) or 0),
}

df_ep_j1 = pd.DataFrame([
    {
        'Process'   : nom,
        'Volume_m3' : round(val, 0),
        'Date_J1'   : date_j1_str
    }
    for nom, val in ep_process_j1.items()
    if val > 0
])

print()
print("EP par process J-1 :")
total_ep_j1 = df_ep_j1['Volume_m3'].sum()
for _, row in df_ep_j1.iterrows():
    pct = row['Volume_m3'] / total_ep_j1 * 100
    print(f"  {row['Process']:<20} : {row['Volume_m3']:>7.0f} m³ ({pct:.1f}%)")
print(f"  {'TOTAL':<20} : {total_ep_j1:>7.0f} m³")

=== DONNÉES J-1 : 2026-05-20 ===
EI par process J-1 :
  Peinture             :     378 m³ (28.1%)
  STEP                 :     306 m³ (22.8%)
  AEB                  :      65 m³ (4.8%)
  TC                   :      15 m³ (1.1%)
  Des TAR              :     163 m³ (12.1%)
  TOUR UE              :     107 m³ (8.0%)
  BD1&2                :       8 m³ (0.6%)
  ST                   :      49 m³ (3.6%)
  Bouclier             :       5 m³ (0.4%)
  Incendie             :     159 m³ (11.8%)
  Autres Process       :      90 m³ (6.7%)
  TOTAL                :    1345 m³

EP par process J-1 :
  Sanitaires           :     341 m³ (76.3%)
  Cantines             :      91 m³ (20.4%)
  IFMIA                :       1 m³ (0.2%)
  Arrosage             :      14 m³ (3.1%)
  TOTAL                :     447 m³


In [42]:
# ── EXPORT VERS GOOGLE SHEETS ────────────────────────────────
import time

# Onglet EI_Process_J1
try:
    ws_ei_j1 = sheet.worksheet("EI_Process_J1")
    print("\nOnglet EI_Process_J1 trouvé ✓")
except gspread.exceptions.WorksheetNotFound:
    ws_ei_j1 = sheet.add_worksheet(
        title="EI_Process_J1", rows=15, cols=4
    )
    print("\nOnglet EI_Process_J1 créé ✓")

time.sleep(1)
data_ei_j1 = [df_ei_j1.columns.tolist()] + df_ei_j1.values.tolist()
ws_ei_j1.clear()
ws_ei_j1.update(data_ei_j1)
print(f"EI_Process_J1 exporté : {len(df_ei_j1)} process ✓")

time.sleep(2)

# Onglet EP_Process_J1
try:
    ws_ep_j1 = sheet.worksheet("EP_Process_J1")
    print("Onglet EP_Process_J1 trouvé ✓")
except gspread.exceptions.WorksheetNotFound:
    ws_ep_j1 = sheet.add_worksheet(
        title="EP_Process_J1", rows=10, cols=4
    )
    print("Onglet EP_Process_J1 créé ✓")

time.sleep(1)
data_ep_j1 = [df_ep_j1.columns.tolist()] + df_ep_j1.values.tolist()
ws_ep_j1.clear()
ws_ep_j1.update(data_ep_j1)
print(f"EP_Process_J1 exporté : {len(df_ep_j1)} process ✓")

print()
print("Google Sheets contient maintenant 7 onglets :")
print("  Feuille 1       → données historiques")
print("  KPIs_Dashboard  → KPIs calculés")
print("  Predictions     → prévisions 7 jours")
print("  J1_Data         → EI/EP totaux J-1")
print("  EI_Process_YTD  → EI par process YTD")
print("  EP_Process_YTD  → EP par process YTD")
print("  EI_Process_J1   → EI par process J-1  ← nouveau")
print("  EP_Process_J1   → EP par process J-1  ← nouveau")


Onglet EI_Process_J1 trouvé ✓
EI_Process_J1 exporté : 11 process ✓
Onglet EP_Process_J1 trouvé ✓
EP_Process_J1 exporté : 4 process ✓

Google Sheets contient maintenant 7 onglets :
  Feuille 1       → données historiques
  KPIs_Dashboard  → KPIs calculés
  Predictions     → prévisions 7 jours
  J1_Data         → EI/EP totaux J-1
  EI_Process_YTD  → EI par process YTD
  EP_Process_YTD  → EP par process YTD
  EI_Process_J1   → EI par process J-1  ← nouveau
  EP_Process_J1   → EP par process J-1  ← nouveau


In [43]:
import math

def barre_kpi(valeur, objectif=1.25, max_val=2.0):
    """Génère une barre de progression ultra-compatible via Tableaux HTML."""
    import math
    if valeur is None or (isinstance(valeur, float) and math.isnan(valeur)):
        return '<div style="font-size:10px; color:#A32D2D; margin-top:2px;">Donnée indisponible</div>'

    valeur_propre = max(0, valeur)
    pct = min(int((valeur_propre / max_val) * 100), 100)
    pct_vide = 100 - pct
    
    # Position de l'objectif en %
    pct_obj = min(int((objectif / max_val) * 100), 100)
    
    couleur = "#A32D2D" if valeur_propre > objectif * 1.15 else \
              "#854F0B" if valeur_propre > objectif else "#27500A"

    return f"""
    <div style="margin-top:5px; position:relative; width:100%; height:14px;">
        <!-- BARRE DE PROGRESSION (Tableau) -->
        <table width="100%" cellspacing="0" cellpadding="0" border="0" style="height:8px; background-color:#eeeeee; border-radius:4px; border-collapse:collapse;">
            <tr>
                <td width="{pct}%" bgcolor="{couleur}" style="height:8px; border-radius:4px 0 0 4px;"></td>
                <td width="{pct_vide}%" bgcolor="#eeeeee" style="height:8px; border-radius:0 4px 4px 0;"></td>
            </tr>
        </table>
        
        <!-- CURSEUR OBJECTIF (Placé avec un tableau invisible par-dessus) -->
        <table width="100%" cellspacing="0" cellpadding="0" border="0" style="position:absolute; top:-3px; left:0; pointer-events:none;">
            <tr>
                <td width="{pct_obj}%" style="border-right:2px solid orange; height:14px;"></td>
                <td width="{100 - pct_obj}%"></td>
            </tr>
        </table>
    </div>
    <div style="font-size:10px; color:#888; margin-top:2px;">▲ Objectif {objectif}</div>
    """

def construire_email_html(analyse, predictions_7j, kpis):
    # ── BLOC SHAP ─────────────────────────────────────────────
    lignes_shap = ""
    for c in analyse['causes']:
        couleur_shap = "#A32D2D" if c['shap'] > 0 else "#27500A"
        lignes_shap += f"""
        <tr>
            <td style="padding:7px 12px;">{c['feature']}</td>
            <td style="padding:7px 12px; text-align:center;">{c['valeur']:.1f}</td>
            <td style="padding:7px 12px; text-align:center; color:{couleur_shap}; font-weight:500;">
                {c['shap']:+.3f}
            </td>
            <td style="padding:7px 12px;">{c['direction']} le KPI</td>
        </tr>"""

    # ── BLOC PRÉVISIONS 7 JOURS ───────────────────────────────
    lignes_prev = ""
    jours_fr = {'Monday':'Lundi','Tuesday':'Mardi','Wednesday':'Mercredi','Thursday':'Jeudi','Friday':'Vendredi','Saturday':'Samedi','Sunday':'Dimanche'}
    mois_fr = {1:'Jan',2:'Fév',3:'Mar',4:'Avr',5:'Mai',6:'Jun',7:'Jul',8:'Aoû',9:'Sep',10:'Oct',11:'Nov',12:'Déc'}

    for _, row in predictions_7j.iterrows():
        if "Alerte" in row['statut']:
            bg_row, bg_kpi, icone = "#FCEBEB", "#A32D2D", "🔴"
            msg = "Consommation prévue trop élevée"
        elif "Attention" in row['statut']:
            bg_row, bg_kpi, icone = "#FAEEDA", "#854F0B", "🟡"
            msg = "Proche du seuil — à surveiller"
        else:
            bg_row, bg_kpi, icone = "#EAF3DE", "#27500A", "🟢"
            msg = "Consommation prévue normale"

        nom_jour = jours_fr.get(row['date'].strftime('%A'), row['date'].strftime('%A'))
        date_str = f"{nom_jour} {row['date'].strftime('%d')} {mois_fr.get(row['date'].month, '')}"
        
        kpi_val = max(0, row['kpi_predit'])
        pct = min(int((kpi_val / 2.0) * 100), 100)
        pct_obj = min(int((1.25 / 2.0) * 100), 100)

        lignes_prev += f"""
        <tr style="border-bottom:1px solid #e0e0e0;">
          <td style="padding:12px 14px; background:{bg_row}; border-left:4px solid {bg_kpi}; width:130px;">
            <div style="font-weight:600; font-size:13px; color:#222;">{date_str}</div>
          </td>
          <td style="padding:12px 14px; background:white; width:200px;">
            <div style="font-size:18px; font-weight:700; color:{bg_kpi};">{kpi_val:.2f} <span style="font-size:12px; font-weight:400; color:#666;">m³/véh</span></div>
            <div style="margin-top:6px; background:#eee; border-radius:4px; height:8px; position:relative;">
              <div style="width:{pct}%; background:{bg_kpi}; height:8px; border-radius:4px;"></div>
              <div style="position:absolute; left:{pct_obj}%; top:-3px; width:2px; height:14px; background:orange;"></div>
            </div>
          </td>
          <td style="padding:12px 14px; background:white; font-size:13px;">{icone} {msg}</td>
        </tr>"""

    # ── CONSTRUCTION DU DASHBOARD KPI ─────────────────────────
    bloc_kpi = f"""
    <div style="padding:16px 24px; background:#f9f9f9; margin-top:2px;">
      <h3 style="margin:0 0 14px; font-size:14px;">📊 Tableau de bord KPI — Consommation eau</h3>
      <table style="width:100%; border-collapse:collapse;">
        <tr>
          <td style="padding:0 6px 0 0; width:25%; vertical-align:top;">
           <div style="background:white; border-radius:8px; padding:12px 14px; border-top:3px solid {kpis['statut_j1']['couleur']};">
              <div style="font-size:11px; color:#888;">Réel J-1</div>
              <div style="font-size:22px; font-weight:700; color:{kpis['statut_j1']['couleur']};">{kpis['kpi_j1']:.3f}</div>
              {barre_kpi(kpis['kpi_j1'])}
            </div>
           
          </td>
          <td style="padding:0 6px; width:25%; vertical-align:top;">
            <div style="background:white; border-radius:8px; padding:12px 14px; border-top:3px solid {kpis['statut_mtd']['couleur']};">
              <div style="font-size:11px; color:#888;">KPI MTD</div>
              <div style="font-size:22px; font-weight:700; color:{kpis['statut_mtd']['couleur']};">{kpis['kpi_mtd']:.3f}</div>
              {barre_kpi(kpis['kpi_mtd'])}
            </div>
          </td>
          <td style="padding:0 6px; width:25%; vertical-align:top;">
            <div style="background:white; border-radius:8px; padding:12px 14px; border-top:3px solid {kpis['statut_ytd']['couleur']};">
              <div style="font-size:11px; color:#888;">KPI YTD</div>
              <div style="font-size:22px; font-weight:700; color:{kpis['statut_ytd']['couleur']};">{kpis['kpi_ytd']:.3f}</div>
              {barre_kpi(kpis['kpi_ytd'])}
            </div>
          </td>
          <td style="padding:0 0 0 6px; width:25%; vertical-align:top;">
            <div style="background:white; border-radius:8px; padding:12px 14px; border-top:3px solid {kpis['statut_annee']['couleur']};">
              <div style="font-size:11px; color:#888;">Prédit 2026</div>
              <div style="font-size:22px; font-weight:700; color:{kpis['statut_annee']['couleur']};">{kpis['kpi_predit_annee']:.3f}</div>
              {barre_kpi(kpis['kpi_predit_annee'])}
            </div>
          </td>
        </tr>
      </table>
    </div>"""

    # ── ASSEMBLAGE FINAL ──────────────────────────────────────
    html = f"""
    <html>
    <body style="font-family:Arial,sans-serif; max-width:700px; margin:auto; color:#222;">
      <div style="background:#1a1a2e; padding:20px 24px; border-radius:8px 8px 0 0;">
        <h2 style="color:white; margin:0;">🌊 Rapport Eau Quotidien</h2>
        <p style="color:#aaa; margin:4px 0 0;">Renault Tanger — {analyse['date'].strftime('%d %B %Y')}</p>
      </div>

      {bloc_kpi}

      <div style="padding:16px 24px; background:#f9f9f9; margin-top:2px;">
        <h3 style="margin:0 0 12px;">🧠 Explication IA (SHAP)</h3>
        <table style="width:100%; border-collapse:collapse; font-size:13px; background:white;">
          <thead><tr style="background:#eee;"><th style="padding:8px;">Variable</th><th>Valeur</th><th>Impact</th><th>Effet</th></tr></thead>
          <tbody>{lignes_shap}</tbody>
        </table>
      </div>

      <div style="padding:16px 24px; margin-top:2px;">
        <h3 style="margin:0 0 12px;">📅 Prévisions 7 jours</h3>
        <table style="width:100%; border-collapse:collapse; box-shadow:0 1px 4px rgba(0,0,0,0.08);">
          <tbody>{lignes_prev}</tbody>
        </table>
      </div>

      <div style="background:#f0f0f0; padding:12px 24px; border-radius:0 0 8px 8px; font-size:11px; color:#888;">
        <p>Rapport généré automatiquement — Système IA Eau | PFE 2026</p>
      </div>
    </body>
    </html>
    """
    return html

# Exécution
email_html = construire_email_html(analyse, predictions_7j, kpis)
print("Email HTML construit ✓")
print(f"Taille : {len(email_html)} caractères")

Email HTML construit ✓
Taille : 14313 caractères


In [44]:
def envoyer_email(html, analyse, config):
    """
    Envoie l'email via Gmail SMTP.
    """
    sujet = (
        f"[🔴 ALERTE EAU] {analyse['date']} — KPI={analyse['kpi_reel']:.3f} m³/véh"
        if analyse['statut'] == 'ANOMALIE'
        else f"[🟢 Rapport Eau] {analyse['date']} — KPI={analyse['kpi_reel']:.3f} m³/véh"
    )

    msg = MIMEMultipart('alternative')
    msg['Subject'] = sujet
    msg['From']    = config['expediteur']        # ← correction 1 : config pas CONFIG_EMAIL
    msg['To']      = ', '.join(config['destinataires'])
    msg.attach(MIMEText(html, 'html'))

    try:
        with smtplib.SMTP(config['smtp_serveur'], config['smtp_port']) as serveur:
            serveur.starttls()
            serveur.login(config['expediteur'], config['mot_de_passe'])
            serveur.sendmail(
                config['expediteur'],
                config['destinataires'],
                msg.as_string()              # ← correction 2 : msg.as_string() pas [msg.as](http://...)
            )
        print(f"✅ Email envoyé avec succès !")
        print(f"   Sujet : {sujet}")
        print(f"   À     : {config['destinataires']}")
    except Exception as e:
        print(f"❌ Erreur envoi : {e}")
        print("   Vérifie le mot de passe application Gmail")

# Test d'envoi — décommente quand prêt
#envoyer_email(email_html, analyse, CONFIG_EMAIL)
print("Fonction prête ✓")

Fonction prête ✓


In [45]:
def envoyer_email(html, analyse, config):
    """
    Envoie l'email via Gmail SMTP.
    """
    sujet = (
        f"[🔴 ALERTE EAU] {analyse['date']} — KPI={analyse['kpi_reel']:.3f} m³/véh"
        if analyse['statut'] == 'ANOMALIE'
        else f"[🟢 Rapport Eau] {analyse['date']} — KPI={analyse['kpi_reel']:.3f} m³/véh"
    )

    msg = MIMEMultipart('alternative')
    msg['Subject'] = sujet
    msg['From']    = config['expediteur']
    msg['To']      = ', '.join(config['destinataires'])
    msg.attach(MIMEText(html, 'html'))

    try:
        with smtplib.SMTP(config['smtp_serveur'], config['smtp_port']) as serveur:
            serveur.starttls()
            serveur.login(config['expediteur'], config['mot_de_passe'])
            serveur.sendmail(
                config['expediteur'],
                config['destinataires'],
                msg.as_string()
            )
        print(f"✅ Email envoyé avec succès !")
        print(f"   Sujet : {sujet}")
        print(f"   À     : {config['destinataires']}")
    except Exception as e:
        print(f"❌ Erreur envoi : {e}")
        print("   Vérifie le mot de passe application Gmail")

# Test d'envoi — décommente quand prêt
#envoyer_email(email_html, analyse, CONFIG_EMAIL)
print("Fonction prête ✓")
print("→ Décommente la dernière ligne pour envoyer l'email")

Fonction prête ✓
→ Décommente la dernière ligne pour envoyer l'email


In [46]:
# Sauvegarder l'email en HTML pour le prévisualiser dans le navigateur
with open("../outputs/email_rapport_eau.html", "w", encoding="utf-8") as f:
    f.write(email_html)

print("Email sauvegardé → ../outputs/email_rapport_eau.html")
print()
print("Pour prévisualiser :")
print("  Double-clique sur le fichier email_rapport_eau.html")
print("  Il s'ouvre dans ton navigateur exactement comme il sera reçu")

Email sauvegardé → ../outputs/email_rapport_eau.html

Pour prévisualiser :
  Double-clique sur le fichier email_rapport_eau.html
  Il s'ouvre dans ton navigateur exactement comme il sera reçu


In [47]:
def rapport_quotidien_complet():
    """
    Fonction principale appelée chaque matin par le cron.
    Regroupe tout : analyse + prédictions + email.
    """
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Démarrage rapport quotidien...")

    # 1. Analyser le jour courant
    analyse = analyser_jour_courant(df_xgb, shap_values, model_xgb)
    print(f"  Statut : {analyse['statut_emoji']} {analyse['statut']}")

    # 2. Prédictions 7 jours
    predictions = predire_7_jours(model_prophet, df_prod)
    print(f"  Prédictions 7j générées ✓")

    # 3. Construire email
    html = construire_email_html(analyse, predictions,kpis)
    print(f"  Email HTML construit ✓")

    # 4. Envoyer seulement si anomalie OU heure = 8h
    heure_actuelle = datetime.now().hour
    if analyse['statut'] == 'ANOMALIE':
        print("  ⚠️  Anomalie détectée → envoi immédiat")
        envoyer_email(html, analyse, CONFIG_EMAIL)
    elif heure_actuelle == 8:
        print("  📧 Rapport matinal → envoi")
        envoyer_email(html, analyse, CONFIG_EMAIL)
    else:
        print("  ℹ️  Pas d'anomalie + heure != 8h → email non envoyé")
        print("      (sauvegardé dans outputs/email_rapport_eau.html)")

    return analyse, predictions

# Lancer le rapport
#analyse_finale, pred_finale = rapport_quotidien_complet()

In [48]:
# ── TEST END-TO-END COMPLET ──────────────────────────────────
from datetime import datetime
import pandas as pd
import os

print("=" * 60)
print(f" TEST END-TO-END — {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

resultats = []

# ── TEST 1 : Fichier Excel accessible ───────────────────────
print("\n[TEST 1] Fichier Excel...")
fichier_excel = "../data/Synthèse_Eaux_2026_VF_(2).xlsm"
if os.path.exists(fichier_excel):
    taille = os.path.getsize(fichier_excel) / 1024
    print(f"  ✅ Fichier trouvé ({taille:.0f} Ko)")
    resultats.append(("Fichier Excel", "✅ OK"))
else:
    print(f"  ❌ Fichier introuvable : {fichier_excel}")
    resultats.append(("Fichier Excel", "❌ ERREUR"))

# ── TEST 2 : Dataset CSV à jour ──────────────────────────────
print("\n[TEST 2] Dataset CSV...")
fichier_csv = "../outputs/dataset_eau_propre.csv"
if os.path.exists(fichier_csv):
    df = pd.read_csv(fichier_csv, parse_dates=['Date'])
    derniere_date = df['Date'].max().date()
    nb_lignes = len(df)
    print(f"  ✅ Dataset trouvé : {nb_lignes} lignes")
    print(f"  ✅ Dernière date  : {derniere_date}")
    resultats.append(("Dataset CSV", f"✅ OK — {nb_lignes} lignes jusqu'au {derniere_date}"))
else:
    print(f"  ❌ Dataset introuvable")
    resultats.append(("Dataset CSV", "❌ ERREUR"))

# ── TEST 3 : Modèles ML chargés ──────────────────────────────
print("\n[TEST 3] Modèles ML...")
import joblib
try:
    m_xgb = joblib.load("../models/model_xgboost.pkl")
    print(f"  ✅ XGBoost chargé ({m_xgb.n_estimators} arbres)")
    resultats.append(("XGBoost", "✅ OK"))
except Exception as e:
    print(f"  ❌ XGBoost : {e}")
    resultats.append(("XGBoost", f"❌ {e}"))

try:
    m_prophet = joblib.load("../models/model_prophet.pkl")
    print(f"  ✅ Prophet chargé")
    resultats.append(("Prophet", "✅ OK"))
except Exception as e:
    print(f"  ❌ Prophet : {e}")
    resultats.append(("Prophet", f"❌ {e}"))

# ── TEST 4 : Google Sheets accessible ───────────────────────
print("\n[TEST 4] Google Sheets...")
try:
    onglets = [ws.title for ws in sheet.worksheets()]
    print(f"  ✅ Sheets accessible")
    print(f"  ✅ Onglets trouvés : {onglets}")
    onglets_requis = [
        "Feuille 1", "KPIs_Dashboard",
        "Predictions", "J1_Data",
        "EI_Process_YTD", "EP_Process_YTD",
        "EI_Process_J1", "EP_Process_J1"
    ]
    for ong in onglets_requis:
        if ong in onglets:
            print(f"    ✅ {ong}")
        else:
            print(f"    ❌ {ong} — MANQUANT")
    resultats.append(("Google Sheets", "✅ OK"))
except Exception as e:
    print(f"  ❌ Sheets : {e}")
    resultats.append(("Google Sheets", f"❌ {e}"))

# ── TEST 5 : KPIs calculés correctement ─────────────────────
print("\n[TEST 5] KPIs...")
try:
    assert kpis['kpi_ytd'] > 0,    "KPI YTD = 0"
    assert kpis['kpi_j1']  > 0,    "KPI J-1 = 0"
    assert kpis['kpi_mtd'] > 0,    "KPI MTD = 0"
    assert not pd.isna(kpis['kpi_ytd']), "KPI YTD = NaN"
    assert not pd.isna(kpis['kpi_j1']),  "KPI J-1 = NaN"
    print(f"  ✅ KPI YTD  : {kpis['kpi_ytd']:.3f} m³/véh")
    print(f"  ✅ KPI MTD  : {kpis['kpi_mtd']:.3f} m³/véh")
    print(f"  ✅ KPI J-1  : {kpis['kpi_j1']:.3f} m³/véh")
    print(f"  ✅ Prédit   : {kpis['kpi_predit_annee']:.3f} m³/véh")
    resultats.append(("KPIs", "✅ OK"))
except AssertionError as e:
    print(f"  ❌ KPIs : {e}")
    resultats.append(("KPIs", f"❌ {e}"))

# ── TEST 6 : Prédictions Prophet ─────────────────────────────
print("\n[TEST 6] Prédictions 7 jours...")
try:
    assert len(predictions_7j) == 7, f"Seulement {len(predictions_7j)} jours prédits"
    assert (predictions_7j['kpi_predit'] > 0).all(), "Valeurs négatives détectées"
    print(f"  ✅ 7 jours prédits")
    for _, row in predictions_7j.iterrows():
        print(f"    {row['date'].strftime('%d/%m')} : {row['kpi_predit']:.3f} {row['statut']}")
    resultats.append(("Prédictions", "✅ OK"))
except AssertionError as e:
    print(f"  ❌ Prédictions : {e}")
    resultats.append(("Prédictions", f"❌ {e}"))

# ── TEST 7 : SHAP explication ────────────────────────────────
print("\n[TEST 7] SHAP...")
try:
    assert len(analyse['causes']) == 3, "Pas 3 causes SHAP"
    assert analyse['statut'] in ['NORMAL','ATTENTION','ANOMALIE']
    print(f"  ✅ Statut     : {analyse['statut_emoji']} {analyse['statut']}")
    print(f"  ✅ 3 causes SHAP identifiées :")
    for c in analyse['causes']:
        print(f"    → {c['feature']} : {c['direction']} le KPI (SHAP={c['shap']:+.3f})")
    resultats.append(("SHAP", "✅ OK"))
except AssertionError as e:
    print(f"  ❌ SHAP : {e}")
    resultats.append(("SHAP", f"❌ {e}"))

# ── TEST 8 : Email HTML construit ───────────────────────────
print("\n[TEST 8] Email HTML...")
try:
    assert len(email_html) > 1000, "Email trop court"
    assert "Rapport Eau" in email_html, "Titre manquant"
    assert "SHAP" in email_html, "Section SHAP manquante"
    assert "Prévisions" in email_html, "Section prévisions manquante"
    print(f"  ✅ Email construit ({len(email_html)} caractères)")
    resultats.append(("Email HTML", "✅ OK"))
except AssertionError as e:
    print(f"  ❌ Email : {e}")
    resultats.append(("Email HTML", f"❌ {e}"))

# ── TEST 9 : Envoi email (test sans envoyer) ─────────────────
print("\n[TEST 9] Configuration email...")
try:
    assert CONFIG_EMAIL['expediteur'] != '', "Expéditeur vide"
    assert len(CONFIG_EMAIL['destinataires']) > 0, "Pas de destinataire"
    assert CONFIG_EMAIL['mot_de_passe'] != '', "Mot de passe vide"
    print(f"  ✅ Expéditeur    : {CONFIG_EMAIL['expediteur']}")
    print(f"  ✅ Destinataires : {CONFIG_EMAIL['destinataires']}")
    resultats.append(("Config email", "✅ OK"))
except AssertionError as e:
    print(f"  ❌ Config email : {e}")
    resultats.append(("Config email", f"❌ {e}"))

# ── RÉSUMÉ FINAL ─────────────────────────────────────────────
print()
print("=" * 60)
print(" RÉSUMÉ DES TESTS")
print("=" * 60)
nb_ok    = sum(1 for _, r in resultats if "✅" in r)
nb_err   = sum(1 for _, r in resultats if "❌" in r)

for test, resultat in resultats:
    print(f"  {test:<20} : {resultat}")

print()
print(f"  Résultat : {nb_ok}/{len(resultats)} tests réussis")
if nb_err == 0:
    print("  🎉 TOUS LES TESTS PASSENT — Système prêt pour la production !")
else:
    print(f"  ⚠️  {nb_err} test(s) échoué(s) — à corriger avant déploiement")
print("=" * 60)

 TEST END-TO-END — 2026-05-21 22:52:44

[TEST 1] Fichier Excel...
  ✅ Fichier trouvé (580 Ko)

[TEST 2] Dataset CSV...
  ✅ Dataset trouvé : 140 lignes
  ✅ Dernière date  : 2026-05-20

[TEST 3] Modèles ML...
  ✅ XGBoost chargé (100 arbres)
  ✅ Prophet chargé

[TEST 4] Google Sheets...
  ✅ Sheets accessible
  ✅ Onglets trouvés : ['Feuille 1', 'Predictions', 'KPIs_Dashboard', 'EI_EP_YTD', 'J1_Data', 'EI_Process_YTD', 'EP_Process_YTD', 'EI_Process_J1', 'EP_Process_J1', 'Historique_J1', 'Peinture_J1', 'Peinture_Gauche', 'Peinture_Centre', 'Peinture_Droite']
    ✅ Feuille 1
    ✅ KPIs_Dashboard
    ✅ Predictions
    ✅ J1_Data
    ✅ EI_Process_YTD
    ✅ EP_Process_YTD
    ✅ EI_Process_J1
    ✅ EP_Process_J1

[TEST 5] KPIs...
  ✅ KPI YTD  : 1.278 m³/véh
  ✅ KPI MTD  : 1.296 m³/véh
  ✅ KPI J-1  : 1.189 m³/véh
  ✅ Prédit   : 1.278 m³/véh

[TEST 6] Prédictions 7 jours...
  ❌ Prédictions : Seulement 6 jours prédits

[TEST 7] SHAP...
  ✅ Statut     : 🟢 NORMAL
  ✅ 3 causes SHAP identifiées :
    → P